# NumiNaMath Dataset Evaluation Notebook

This notebook demonstrates how to:
1. Load the NumiNaMath dataset from local files or HuggingFace.
2. Preprocess samples to extract the problem and boxed solution.
3. Call a frontier model API (e.g., GPT) using an API key from environment variables.
4. Run inference on 5 random test samples.
5. Compute cross-entropy between ground truth and prediction.


## 1. Import Required Libraries

We will use `datasets` for data loading, `re` for regex extraction, `os` for environment variables, and `random` for sampling.

In [1]:
import os
from pathlib import Path

#------------ CACHE ENVIRONMENT ----------------------------------------
# Set cache environment variables BEFORE importing datasets
# or cache env variables will be reset at import and trigger errors
PROJECT_ROOT = Path.cwd().parent  # Go up one level
hf_cache = PROJECT_ROOT / ".cache" / "huggingface"
hf_cache.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_cache)
os.environ["HF_DATASETS_CACHE"] = str(hf_cache / "datasets")
os.environ["HF_HUB_CACHE"] = str(hf_cache)
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache / "transformers")

In [2]:
# -------------- other imports -----------------------------------------
import re
import random
from datasets import load_dataset, Dataset, load_from_disk
import requests
import numpy as np
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig
import torch
from pprint import pprint
from tqdm import tqdm

## 2. Load the NumiNaMath Dataset

Try to load the dataset from the local directory. If not found, download from HuggingFace.

In [3]:
# Define local dataset path
local_dataset_path = Path('../datasets/test/data-00000-of-00001.arrow').resolve()

# dataset ID
HF_DATASET_ID = "AI-MO/NuminaMath-TIR"
PROJECT_ROOT = Path.cwd().parent  # Go up one level
DATASET_DIR = PROJECT_ROOT / "datasets"

# logic
if local_dataset_path.exists():
    print(f"Loading dataset from local path: {local_dataset_path}")
    # dataset = Dataset.load_from_disk(str(local_dataset_path.parent.parent))
    dataset = load_from_disk(str(DATASET_DIR))
    # test_split = dataset['test']
else:
    print("Local dataset not found. Downloading from HuggingFace...")
    dataset = load_dataset(HF_DATASET_ID)
    # test_split = dataset['test']

# print(f"Loaded {len(test_split)} test samples.")

Loading dataset from local path: /home/benjamin.deporte/GenAI_Math/datasets/test/data-00000-of-00001.arrow


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 72441
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 99
    })
})

In [5]:
# select x % from the training dataset to validate the pipeline
train_dataset = dataset['train'].select(range(int(0.01*len(dataset['train']))))
test_dataset = dataset['test']

print(f"Train size:          {len(train_dataset)}")
print(f"Validation size:     {len(test_dataset)}")

Train size:          724
Validation size:     99


In [6]:
train_dataset

Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 724
})

Dataset Brut

In [7]:
N_SAMPLES = 5
SAMPLE_SEED = 123
MAX_NEW_TOKENS = 2048

rng = np.random.default_rng(SAMPLE_SEED)
n_samples = min(N_SAMPLES, len(test_dataset))
sampled_indices = rng.choice(len(test_dataset), size=n_samples, replace=False).tolist()

for idx in sampled_indices:
    sample = test_dataset[idx]
    print(f"-"*50)
    pprint(sample)

--------------------------------------------------
{'messages': [{'content': 'Given the function $f(x)= \\begin{cases} x+2 & (x '
                          '\\leqslant -2) \\\\ 2^{x} & (-2 < x < 3) \\\\ \\ln '
                          'x & (x \\geqslant 3) \\end{cases}$, find '
                          '$f(f(-2))=$ _______.',
               'role': 'user'},
              {'content': 'To solve this problem, we need to follow these '
                          'steps:\n'
                          '\n'
                          '1. Understand the piecewise function \\( f(x) \\):\n'
                          '   \\[\n'
                          '   f(x)= \\begin{cases} \n'
                          '   x + 2 & \\text{if } x \\leq -2 \\\\\n'
                          '   2^x & \\text{if } -2 < x < 3 \\\\\n'
                          '   \\ln x & \\text{if } x \\geq 3 \n'
                          '   \\end{cases}\n'
                          '   \\]\n'
                          '\n'
      

In [8]:
# first utility function,
# extract the string between the opening \boxed{ and the corresponding closing }

# def extract_boxed(solution: str) -> str:
#     """ 
#     count the opening { and closing } to extract the right 'solution'
#     string from the \boxed{ ... } expression
#     """
#     start = solution.find(r'\boxed{')
#     if start == -1:
#         # raise ValueError("Boxed solution not found in solution field.")*
#         return ""
        
#     # Move to the opening brace
#     brace_pos = start + len(r'\boxed{') - 1
#     depth = 0
#     for i in range(brace_pos, len(solution)):
#         if solution[i] == '{':
#             depth += 1
#         elif solution[i] == '}':
#             depth -= 1
#             if depth == 0:
#                 return solution[brace_pos + 1:i]  # content between the braces
#     raise ValueError("Unmatched braces in \\boxed{} expression.")


# from https://gist.github.com/twobob/7c3722fdb682c1f0b247d8b617502887
def extract_boxed(text: str) -> str | None:
    """
    Extracts the content inside the first occurrence of a LaTeX-style \boxed{...} in the given text.
    Supports nested curly braces.
    Returns the extracted answer as a string, or None if no properly balanced \boxed{...} is found.
    """
    start_marker = r'\boxed{'
    start_idx = text.find(start_marker)
    if start_idx == -1:
        return ""

    # Position immediately after the opening brace of \boxed{
    content_start = start_idx + len(start_marker)
    brace_count = 1
    i = content_start

    # Traverse the text to find the matching closing brace, accounting for nested braces
    while i < len(text) and brace_count > 0:
        char = text[i]
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
        i += 1

    # If the braces are unbalanced, return None
    if brace_count != 0:
        raise ValueError("Unmatched braces in \\boxed{} expression.")
        # return None

    # Extract the content inside the outermost \boxed{...}
    answer = text[content_start:i - 1]
    return answer.strip()

# second utility function
# take a point of the dataset, format it into a problem, solution, solution set.

def extract_problem_and_boxed_solution(sample):
    """
    Utility function to extract problem, solution, and boxed solution from a sample in the test split of NuminaMath
    Inputs :
        - sample (dict, with keys:
            'problem' : str,
            'solution' : str,
            'messages' : list of dict,
        }
        the value sample['solution'] contains the chain of thoughts and the ultimate solution in \boxed{ .. }
    Returns:
        - problem (str) : the problem
        - solution (str) : the whole ground truth response
        - boxed_solution (str) : the extract between \boxed{ and }, if any, None otherwise
    """
    
    problem = sample.get('problem', None)
    solution = sample.get('solution', None)
    if problem is None or solution is None:
        raise ValueError("Sample missing 'problem' or 'solution' field.")
    
    # Match everything between \\boxed{ and the corresponding closing }
    boxed_solution = extract_boxed(solution)
    
    return problem, solution, boxed_solution

In [9]:
# Preprocess the dataset pour avoir le format qui va bien pour le SFT

def preprocess_function(example):
    
    problem, solution, boxed_solution = extract_problem_and_boxed_solution(example)
    
    return {
        "prompt": [{"role": "user", "content": problem}],
        "completion": [
            {"role": "assistant", "content": f"<think>{solution}</think><answer>{boxed_solution}</answer>"}
        ],
    }

In [10]:
preprocessed_train_dataset = train_dataset.map(preprocess_function, remove_columns=["problem", "solution", "messages"])
preprocessed_test_dataset = test_dataset.map(preprocess_function, remove_columns=["problem", "solution", "messages"])

Dataset préprocessé

In [11]:
for idx in sampled_indices:
    sample = preprocessed_test_dataset[idx]
    print(f"-"*50)
    pprint(sample)

--------------------------------------------------
{'completion': [{'content': '<think>To solve this problem, we need to follow '
                            'these steps:\n'
                            '\n'
                            '1. Understand the piecewise function \\( f(x) '
                            '\\):\n'
                            '   \\[\n'
                            '   f(x)= \\begin{cases} \n'
                            '   x + 2 & \\text{if } x \\leq -2 \\\\\n'
                            '   2^x & \\text{if } -2 < x < 3 \\\\\n'
                            '   \\ln x & \\text{if } x \\geq 3 \n'
                            '   \\end{cases}\n'
                            '   \\]\n'
                            '\n'
                            '2. Find \\( f(-2) \\) using the piecewise '
                            'function definition.\n'
                            '3. Use the result from step 2 to find \\( '
                            'f(f(-2)) \\).\n'
          

# 3. Load model, perform LoRA

In [12]:
# utility code - idem, télécharge le modèle si pas déjà présent sur le disque
# et le sauve localement, ainsi que le tokenizer associé

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = Path("../models/qwen3-0.6B-SFT").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
is_empty = not any(MODEL_DIR.iterdir())

if is_empty:
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,   # ou torch.float16 selon ton GPU
        device_map="auto"
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        dtype=torch.bfloat16,
        device_map="auto"
    )

/home/benjamin.deporte/GenAI_Math/models/qwen3-0.6B-SFT is not empty -> loading model/tokenizer from disk...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [13]:
# utility function pour calculer le nombre de paramètres trainables
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [14]:
base_model = model

# Before LoRA (full model)
full_trainable, full_total = count_params(base_model)

print(f"Modèle : {MODEL_ID}, full FT trainable: {full_trainable:,}")

Modèle : Qwen/Qwen3-0.6B, full FT trainable: 596,049,920


In [15]:
# using PEFT and LoRA pour réduire le besoin en VRAM
RANG=4

peft_config=LoraConfig(
    task_type="CAUSAL_LM",   # decrit quel type de modèle on adapte
    inference_mode=False, # ordonne que les paramètres soient entrainables
    r=RANG, # rang !
    lora_alpha=2*RANG, # scaling factor pour l'update LoRA
    lora_dropout=0.05, # dropout pour le LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # LoRA sur les têtes d'attention
)

lora_model=get_peft_model(base_model, peft_config)

In [16]:
# After LoRA wrapping
lora_trainable, lora_total = count_params(lora_model)

reduction_vs_full_ft = 100 * (1 - (lora_trainable / full_trainable))
trainable_pct_of_total = 100 * (lora_trainable / lora_total)

print(f"Modèle : {MODEL_ID}, full FT trainable: {full_trainable:,}")
print(f"LoRA trainable:    {lora_trainable:,}")
print(f"Reduction:         {reduction_vs_full_ft:.2f}%")
print(f"LoRA trainable %:  {trainable_pct_of_total:.4f}%")

Modèle : Qwen/Qwen3-0.6B, full FT trainable: 596,049,920
LoRA trainable:    1,146,880
Reduction:         99.81%
LoRA trainable %:  0.1920%


# 4. Evaluation metrics

In [17]:
# utility function
def preprocess_logits_for_metrics(logits, labels):
    # logits est typiquement (batch, seq_len, vocab_size)
    if isinstance(logits, tuple):
        # some configurations return logits avec shape (logits, hidden_states)
        # dans ce cas, on ne garde que logits...
        logits = logits[0] # shape (batch_size, seq_len, vocab_size)
    # renvoie (batch_size, seq_len) avec le predicted token id
    return logits.argmax(dim=-1)

# Optional metric: token-level accuracy (ignores -100 labels, ie padding tokens)
# compare les deux chaînes de tokens, token par token
def compute_metrics(eval_pred):
    preds, labels = eval_pred  # numpy arrays
    # next-token shift
    preds = preds[:, :-1].reshape(-1)
    labels = labels[:, 1:].reshape(-1)

    # gère le edge case où tous les tokens "labels" sont -100
    mask = labels != -100
    if mask.sum() == 0:
        return {"token_accuracy": 0.0}

    # compte 1 pour les bons tokens prédits, et la moyenne
    acc = (preds[mask] == labels[mask]).astype(np.float32).mean().item()
    return {"token_accuracy": acc}

# 5. Training

In [18]:
OUTPUT_DIR = PROJECT_ROOT / "outputs/sft-numinamath"
LOGGING_DIR = PROJECT_ROOT / "logs"

# Enable eval during training
sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=8,      # important pour ma pauvre 3080 Ti 12 Go
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    
    # logging strategy
    logging_strategy="steps",
    logging_dir=str(LOGGING_DIR),
    logging_steps=10,
    # max_seq_length=512,
    gradient_checkpointing=True,
    bf16=True,   
    
    # evaluation strategy
    eval_strategy="steps",             # if your version expects evaluation_strategy, use that key instead
    eval_steps=10,
    
    # save model strategy
    save_steps=100,
    save_strategy="steps",
    
    # other
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=8,         # helps eval memory
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
# doc HuggingFace : 
# Expected dataset type and format
# SFT supports both language modeling and prompt-completion datasets. 
# The SFTTrainer is compatible with both standard and conversational dataset formats. 
# When provided with a conversational dataset, the trainer will automatically apply the chat template to the dataset.
# Conversational prompt-completion
# {"prompt": [{"role": "user", "content": "What color is the sky?"}],
# "completion": [{"role": "assistant", "content": "It is blue."}]}

trainer = SFTTrainer(
    model=lora_model,  # use the already reduced model
    args=sft_args,
    train_dataset=preprocessed_train_dataset,
    eval_dataset=preprocessed_test_dataset,
    processing_class=tokenizer,  # optionnel. sinon, SFTTrainer utilisera le tokenizer du modèle par défaut
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [20]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Token Accuracy
10,0.697715,0.577263,0.830359
20,0.579795,0.520917,0.843589
30,0.509910,0.498342,0.849744
40,0.497507,0.488239,0.851301
50,0.458041,0.482308,0.852504
60,0.498990,0.478666,0.853406
70,0.469012,0.476058,0.854732
80,0.477756,0.473872,0.855511
90,0.460828,0.472262,0.855440
100,0.486086,0.470971,0.855847


TrainOutput(global_step=138, training_loss=0.4998907058135323, metrics={'train_runtime': 523.3901, 'train_samples_per_second': 4.15, 'train_steps_per_second': 0.264, 'total_flos': 3767106303295488.0, 'train_loss': 0.4998907058135323})

In [21]:
model_for_eval = trainer.model
model_for_eval.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(


# 6. Evaluation

In [22]:
def parse_completion(completion_text):
    """
    Parse completion text to extract reasoning and final answer.
    Format: <think>REASONING</think><answer>FINAL_ANSWER</answer>

    Returns: (reasoning_text, final_answer_text)
    """
    reasoning = ""
    final_answer = ""

    # Try to extract from <think>...</think> tags
    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()

    # Try to extract from <answer>...</answer> tags
    if "<answer>" in completion_text and "</answer>" in completion_text:
        answer_start = completion_text.find("<answer>") + len("<answer>")
        answer_end = completion_text.find("</answer>")
        final_answer = completion_text[answer_start:answer_end].strip()
    # Fallback: if no <answer> tags, use everything after </think>
    elif "</think>" in completion_text:
        fallback_start = completion_text.find("</think>") + len("</think>")
        final_answer = completion_text[fallback_start:].strip()
    # Last resort: if no tags at all, return the whole text
    else:
        final_answer = completion_text.strip()

    return reasoning, final_answer

In [23]:
N_SAMPLES = 5
SAMPLE_SEED = 123
MAX_NEW_TOKENS = 2048

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

rng = np.random.default_rng(SAMPLE_SEED)
n_eval = min(N_SAMPLES, len(test_dataset))
sample_indices = rng.choice(len(test_dataset), size=n_eval, replace=False).tolist()

print(f"Sampling {n_eval} examples from validation set of size {len(test_dataset)}")

Sampling 5 examples from validation set of size 99


In [24]:
results = []
sampled_indices = random.sample(range(len(test_dataset)), 3)

for idx in sampled_indices:
    sample = test_dataset[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    
    results.append({
        'problem': problem,
        'solution': solution,
        'ground_truth': boxed_solution,
    })
    
for i, res in enumerate(results):
    print(f"sample {i+1}", "-"*40)
    print(f"problem: {res['problem']}")
    print(f"full solution: {res['solution']}")
    print(f"ground truth: {res['ground_truth']}\n\n")

sample 1 ----------------------------------------
problem: 
Let's call the distance between two triangles the smallest of the distances. Is it possible to arrange five triangles on the plane so that the distance between any two of them is equal to the sum of the radii of their circumscribed circles?
full solution: Let's solve this problem step-by-step.

### Problem Understanding and Breakdown

Given five triangles on a plane, we need to determine if their arrangement can be such that the distance between any two of them is equal to the sum of the radii of their circumscribed circles.

### Key Concepts

1. **Circumcircle of a Triangle:** The circumcircle of a triangle is the unique circle that passes through all three vertices of the triangle. The radius of this circle is called the circumradius.
2. **Distance Between Triangles:** We are considering the distance between two triangles to be the smallest of the distances between any pair of points, one from each triangle.

### Assumptions

# Run Inference on some Random Test Samples

Randomly select some samples, extract the problem and boxed solution, call the model, and collect predictions.

In [25]:
N_SAMPLES = 5

results = []
sampled_indices = random.sample(range(len(test_dataset)), N_SAMPLES)

for idx in tqdm(sampled_indices):
    sample = test_dataset[idx]
    
    # extrait problème, solution, boxed_solution du sample
    try:
        prompt_completion = preprocess_function(sample)
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue

    try:
        # formate la question posée au format chat
        messages = prompt_completion["prompt"]

        # tokenize la demande
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        
        # forme les inputs
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)

        # genere une réponse
        with torch.no_grad():
            generated = model_for_eval.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        gen_tokens = generated[0][prompt_len:]
        pred_completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        gt_full = f"<think>{solution}</think><answer>{boxed_solution}</answer>"
        gt_reasoning, gt_final_answer = parse_completion(gt_full)
        pred_reasoning, pred_final_answer = parse_completion(pred_completion)

        results.append(
            {
                "idx": int(idx),
                "problem": problem,
                "solution": solution,
                "ground_truth": gt_final_answer,
                "prediction": pred_final_answer,
                "gt_reasoning": gt_reasoning,
                "pred_reasoning": pred_reasoning,
                "gt_full": gt_full,
                "pred_full": pred_completion,
            }
        )
    except Exception as e:
        print(f"Generation failed for sample {idx}: {e}")
        continue

100%|██████████| 5/5 [01:39<00:00, 19.89s/it]


In [26]:
print(f"Built {len(results)} comparisons with parsed components.")

for i, res in enumerate(results):
    print(f"\nSample {i + 1}:")
    # print(f"Res = {res}")
    print(f"Problem: {res['problem']}")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Prediction: {res['prediction']}")
    print('-' * 40, "\n")

Built 5 comparisons with parsed components.

Sample 1:
Problem: The integer 119 is a multiple of:
(A) 2
(B) 3
(C) 5
(D) 7
(E) 11
Ground Truth: 7
Prediction: 7
---------------------------------------- 


Sample 2:
Problem: For a positive integer \( n \), let \( S(n) \) denote the sum of its digits. Find all positive integers \( M \) such that for every positive integer \( k \) not exceeding \( M \), \( S(Mk) = S(M) \).
Ground Truth: 9, 99, 999
Prediction: 1, 2, 3, 4, 5, 6, 7, 8, 9
---------------------------------------- 


Sample 3:
Problem: A cube has all its vertices on the surface of a sphere. If the volume of the sphere is $4 \sqrt {3} \pi$, what is the surface area of the cube?
Ground Truth: 24
Prediction: 12
---------------------------------------- 


Sample 4:
Problem: The sequence $\{a_n\}$ is an arithmetic sequence with the first term $1$ and common difference $3$. If $a_n = 2014$, then the index $n = \boxed{672}$.
Ground Truth: 672
Prediction: 672
----------------------------

# Comparing ground truth and predictions

In [27]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

HF_HOME: /home/benjamin.deporte/GenAI_Math/.cache/huggingface
HF_HUB_CACHE: /home/benjamin.deporte/GenAI_Math/.cache/huggingface
TRANSFORMERS_CACHE: /home/benjamin.deporte/GenAI_Math/.cache/huggingface/transformers


In [28]:
# First attempt : use BERTScore

from bert_score import score

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    gt = res['ground_truth']
    pred = res['prediction']
    print(f"Ground Truth: {gt}")
    print(f"Prediction: {pred}")
    P, R, F1 = score([gt], [pred], lang="en")
    print(f"P = {P.item():.3f}, R = {R.item():.3f}, F1 = {F1.item():.3f}")
    print('-'*40, "\n")

Sample 1:
Ground Truth: 7
Prediction: 7


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 1.000, R = 1.000, F1 = 1.000
---------------------------------------- 

Sample 2:
Ground Truth: 9, 99, 999
Prediction: 1, 2, 3, 4, 5, 6, 7, 8, 9


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 1.000, R = 1.000, F1 = 1.000
---------------------------------------- 

Sample 5:
Ground Truth: \sqrt{20} - \sqrt{5} + \sqrt{\frac{1}{5}} = \sqrt{20} - \sqrt{5} + \frac{1}{\sqrt{5}} = 2\sqrt{5} - \sqrt{5} + \frac{1}{\sqrt{5}} = \sqrt{5}(2 - 1 + \frac{1}{5}) = \sqrt{5}(1 + \frac{1}{5}) = \sqrt{5} \cdot \frac{6}{5} = \frac{6\sqrt{5}}{5} = \boxed{\frac{6}{5}\sqrt{5}}
Prediction: \[
\sqrt{5} + \frac{1}{\sqrt{5}}
\]


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P = 0.867, R = 0.876, F1 = 0.871
---------------------------------------- 

